In [4]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data
df = pd.read_csv('../data/raw/insurance.csv')

# 1. Create Age Bands
df['age_band'] = pd.cut(
    df['age'], 
    bins=[0, 30, 45, 60, 100], 
    labels=['18-30', '31-45', '46-60', '60+']
)

# 2. Create BMI Bands
df['bmi_band'] = pd.cut(
    df['bmi'], 
    bins=[0, 25, 30, 35, 100], 
    labels=['normal', 'overweight', 'obese', 'severely_obese']
)

# Drop original 'age' and 'bmi' if we want to rely solely on bands, 
# but usually, keeping both helps the boosting model!
df.head()

,age,sex,bmi,children,smoker,region,charges,age_band,bmi_band
0,19,female,27.900,0,yes,southwest,16884.92400,18-30,overweight
1,18,male,33.770,1,no,southeast,1725.55230,18-30,obese
2,28,male,33.000,3,no,southeast,4449.46200,18-30,obese
3,33,male,22.705,0,no,northwest,21984.47061,31-45,normal
4,32,male,28.880,0,no,northwest,3866.85520,31-45,overweight


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Define features
cat_features = ['sex', 'smoker', 'region', 'age_band', 'bmi_band']
num_features = ['children', 'age', 'bmi'] # Keep numeric too for better precision

# Setup Preprocessor
preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_features),
    ('num', StandardScaler(), num_features)
])

# Split Data
X = df.drop('charges', axis=1)
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data successfully split and preprocessor ready!")

Data successfully split and preprocessor ready!


In [6]:
import joblib
import os

os.makedirs('../data/processed', exist_ok=True)

# Save the preprocessor and data
joblib.dump(preprocessor, '../models/preprocessor.joblib')
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)